In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json, numpy as np
from datasets import Dataset

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [ ]:
absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"
train_rows = load_jsonl(absa_train_path)
dev_rows = load_jsonl(absa_dev_path)
test_rows = load_jsonl(absa_test_path)

In [ ]:
all_cats = set()
for split in (train_rows, dev_rows, test_rows):
    for r in split:
        for lab in r["labels"]:
            all_cats.add(lab["category"])
CATS = sorted(list(all_cats))
cat2id = {c:i for i,c in enumerate(CATS)}

In [ ]:
def rows_to_multilabel(rows):
    texts = []
    ys = []
    for r in rows:
        text = r["text"]
        vec = np.zeros(len(CATS), dtype=np.float32)
        for lab in r["labels"]:
            c = lab["category"]
            if c in cat2id:
                vec[cat2id[c]] = 1.0
        texts.append(text)
        ys.append(vec)
    return {"text": texts, "labels": ys}

In [ ]:
train_cat = Dataset.from_dict(rows_to_multilabel(train_rows))
dev_cat = Dataset.from_dict(rows_to_multilabel(dev_rows))
test_cat = Dataset.from_dict(rows_to_multilabel(test_rows))

In [ ]:
from transformers import AutoTokenizer
model_name = "aubmindlab/bert-base-arabertv2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_cat = train_cat.map(tok_fn, batched=True)
dev_cat = dev_cat.map(tok_fn, batched=True)
test_cat = test_cat.map(tok_fn, batched=True)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/1564 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/452 [00:00<?, ? examples/s]

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

aspect_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(CATS),
)
aspect_model.config.problem_type = "multi_label_classification"

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score

collator = DataCollatorWithPadding(tokenizer)

def compute_multilabel_metrics(eval_pred, threshold=0.5):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= threshold).astype(int)

    micro = f1_score(labels, preds, average="micro", zero_division=0)
    macro = f1_score(labels, preds, average="macro", zero_division=0)
    return {"micro_f1": micro, "macro_f1": macro}

args = TrainingArguments(
    output_dir="/content/drive/MyDrive/FYP/absa/arabert_aspect_detector",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

aspect_trainer = Trainer(
    model=aspect_model,
    args=args,
    train_dataset=train_cat,
    eval_dataset=dev_cat,
    data_collator=collator,
    compute_metrics=compute_multilabel_metrics,
)

aspect_trainer.train()
aspect_test = aspect_trainer.evaluate(test_cat)
print(aspect_test)

out_dir_aspect = "/content/drive/MyDrive/FYP/absa/arabert_aspect_detector"
aspect_trainer.save_model(out_dir_aspect)

Epoch,Training Loss,Validation Loss,Micro F1,Macro F1
1,0.461779,0.288780,0.563734,0.053890
2,0.317005,0.255594,0.563734,0.053890
3,0.295546,0.249285,0.563734,0.053890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.3167683780193329, 'eval_micro_f1': 0.5127367733507512, 'eval_macro_f1': 0.054364930685650056, 'eval_runtime': 0.246, 'eval_samples_per_second': 1837.252, 'eval_steps_per_second': 60.971, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch, numpy as np

sent_dir = "/content/drive/MyDrive/FYP/absa/bert_arabert_conditioned_absa"
tok_name = "aubmindlab/bert-base-arabertv2"
sent_tok = AutoTokenizer.from_pretrained(tok_name)
sent_model = AutoModelForSequenceClassification.from_pretrained(sent_dir)
sent_model.eval().to("cuda" if torch.cuda.is_available() else "cpu")

LABELS = ["negative", "neutral", "positive", "conflict"]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
import json, os
asp_dir = "/content/drive/MyDrive/FYP/absa/arabert_aspect_detector"
asp_tok = AutoTokenizer.from_pretrained(tok_name)
asp_model = AutoModelForSequenceClassification.from_pretrained(asp_dir)
asp_model.eval().to("cuda" if torch.cuda.is_available() else "cpu")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(64000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

@torch.no_grad()
def predict_aspects_batch(texts, threshold=0.5):
    enc = asp_tok(
        texts,
        truncation=True,
        max_length=256,
        padding=True,
        return_tensors="pt"
    ).to(device)
    logits = asp_model(**enc).logits
    probs = torch.sigmoid(logits).cpu().numpy()

    all_aspects = []
    for p in probs:
        idxs = np.where(p >= threshold)[0].tolist()
        all_aspects.append([CATS[i] for i in idxs])
    return all_aspects

@torch.no_grad()
def predict_sentiment_pairs(categories, texts):
    enc = sent_tok(
        categories,
        texts,
        truncation=True,
        max_length=256,
        padding=True,
        return_tensors="pt"
    ).to(device)
    logits = sent_model(**enc).logits
    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    return [LABELS[i] for i in pred_ids]

def predict_pairs_for_texts(texts, asp_threshold=0.5, batch_size=32):
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_aspects = predict_aspects_batch(batch_texts, threshold=asp_threshold)

        flat_cats, flat_texts, sizes = [], [], []
        for t, aspects in zip(batch_texts, batch_aspects):
            sizes.append(len(aspects))
            for a in aspects:
                flat_cats.append(a)
                flat_texts.append(t)

        flat_sents = []
        if flat_cats:
            flat_sents = predict_sentiment_pairs(flat_cats, flat_texts)

        idx = 0
        for aspects, n in zip(batch_aspects, sizes):
            s = set()
            for j in range(n):
                s.add((aspects[j], flat_sents[idx + j]))
            idx += n
            results.append(s)

    return results

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def norm_pol(p):
    return p.strip().lower()

def gold_pairs_from_row(r):
    g = set()
    for lab in r["labels"]:
        cat = lab["category"]
        pol = norm_pol(lab["polarity"])
        if pol in LABELS:
            g.add((cat, pol))
    return g

def micro_prf(gold_sets, pred_sets):
    tp = fp = fn = 0
    for g, p in zip(gold_sets, pred_sets):
        tp += len(g & p)
        fp += len(p - g)
        fn += len(g - p)

    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    f1   = (2*prec*rec)/(prec+rec) if (prec+rec) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": prec, "recall": rec, "f1": f1}

In [ ]:
test_texts = [r["text"] for r in test_rows]
gold = [gold_pairs_from_row(r) for r in test_rows]
pred = predict_pairs_for_texts(test_texts, asp_threshold=0.5, batch_size=32)

final_metrics = micro_prf(gold, pred)
print(final_metrics)


save_path = "/content/drive/MyDrive/FYP/absa/bert_results/arabert_unconditioned_absa_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump({"micro-f1": final_metrics["f1"], "precision":final_metrics["precision"], "recall":final_metrics["recall"]}, f, ensure_ascii=False, indent=2)

{'tp': 692, 'fp': 212, 'fn': 1466, 'precision': 0.7654867256637168, 'recall': 0.3206672845227062, 'f1': 0.45199216198563036}
